# Semantic Dictionaries: Hierarchical Feature Geometry from a JSON Taxonomy

This tutorial shows how to build a synthetic model whose ground-truth feature vectors are
arranged according to a **semantic dictionary** — a JSON file that defines a forest of
concept trees, where parent–child cosine similarity is controlled by the parameter α.

The geometry rule is:

$$d_{\text{child}} = \alpha\, d_{\text{parent}} + \beta\, d_{\perp}, \quad \alpha^2 + \beta^2 = 1$$

where $d_{\perp}$ is a unit vector orthogonal to $d_{\text{parent}}$ (Gram–Schmidt).
This guarantees $\cos(d_{\text{child}}, d_{\text{parent}}) = \alpha$ exactly.

We use the **MITRE ATLAS Adversarial ML taxonomy** (`feature_hierarchies/mitre_atlas_adversarial_ml.json`)
which ships with this repository. It defines 13 root concepts (tactics), each with a
3-level tree structure. Features not covered by the JSON become free/non-hierarchical
features with mutually orthogonal vectors.

**This notebook runs comfortably on CPU** — the model is small (512 features, 128-dim hidden
space) and training is fast (~2 minutes on CPU).

## Setup

In [1]:
import json
import pathlib
import warnings

import torch

try:
    import google.colab  # type: ignore

    COLAB = True
    %pip install sae-lens
except Exception:
    COLAB = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Path to the JSON taxonomy (relative to this notebook)
REPO_ROOT = pathlib.Path(".").resolve().parent
JSON_PATH = REPO_ROOT / "feature_hierarchies" / "mitre_atlas_adversarial_ml.json"
print(f"JSON path: {JSON_PATH}")

Using device: cpu
JSON path: /Users/bioitx/Documents/GitHub/SAELens/feature_hierarchies/mitre_atlas_adversarial_ml.json


## Inspecting the JSON taxonomy

The semantic dictionary JSON has a top-level `"trees"` array. Each tree is a nested
structure with `label`, `alpha`, `beta`, `mutually_exclusive_children`, and `children`.

Let's load it and count how many features the hierarchy will occupy.

In [2]:
with open(JSON_PATH) as f:
    taxonomy = json.load(f)

print(f"Source:      {taxonomy['source']}")
print(f"Root trees:  {len(taxonomy['trees'])}")
print()
print("Root labels:")
for i, tree in enumerate(taxonomy["trees"]):
    print(f"  {i:2d}. {tree['label']}")

Source:      MITRE ATLAS (Adversarial Threat Landscape for Artificial-Intelligence Systems)
Root trees:  13

Root labels:
   0. Adversarial Evasion
   1. Adversarial Perturbation Crafting
   2. Data Poisoning
   3. Backdoor and Trojan Attacks
   4. Model Extraction and Stealing
   5. Membership Inference
   6. Model Inversion and Data Reconstruction
   7. Prompt Injection and LLM Exploitation
   8. ML Reconnaissance
   9. ML Supply Chain Attack
  10. Adversarial Robustness Degradation
  11. ML Model Access and Initial Compromise
  12. ML Exfiltration


In [3]:
def count_nodes(node: dict) -> int:
    return 1 + sum(count_nodes(c) for c in node.get("children", []))


def tree_depth(node: dict) -> int:
    children = node.get("children", [])
    if not children:
        return 0
    return 1 + max(tree_depth(c) for c in children)


total_hierarchical = sum(count_nodes(t) for t in taxonomy["trees"])
max_depth = max(tree_depth(t) for t in taxonomy["trees"])
sizes = [count_nodes(t) for t in taxonomy["trees"]]

print(f"Total hierarchical features: {total_hierarchical}")
print(f"Max tree depth:              {max_depth}")
print(f"Min / avg / max tree size:   {min(sizes)} / {sum(sizes)/len(sizes):.1f} / {max(sizes)}")

Total hierarchical features: 148
Max tree depth:              2
Min / avg / max tree size:   10 / 11.4 / 13


## Building the synthetic model

We create a `SyntheticModel` with:
- `num_features=512` — the JSON defines ~147 hierarchical features; the remaining ~365
  become free features with mutually orthogonalized vectors
- `hidden_dim=128` — a compact hidden space that still allows meaningful superposition
- `semantic_dictionary_path` set to our JSON — this triggers the semantic geometry initializer

No `orthogonalization` config is needed: the semantic initializer handles free features
automatically by orthogonalizing them via the existing `orthogonalize_embeddings` routine.

In [4]:
from sae_lens.synthetic import SyntheticModel, SyntheticModelConfig, ZipfianFiringProbabilityConfig
from sae_lens.synthetic.hierarchy import HierarchyConfig

NUM_FEATURES = 512
HIDDEN_DIM = 128

cfg = SyntheticModelConfig(
    num_features=NUM_FEATURES,
    hidden_dim=HIDDEN_DIM,
    firing_probability=ZipfianFiringProbabilityConfig(
        exponent=0.5,
        max_prob=0.3,
        min_prob=1e-3,
    ),
    hierarchy=HierarchyConfig(
        semantic_dictionary_path=str(JSON_PATH),
    ),
    orthogonalization=None,  # handled by semantic initializer
    seed=42,
)

model = SyntheticModel(cfg, device=device)

print(f"Features:        {model.cfg.num_features}")
print(f"Hidden dim:      {model.cfg.hidden_dim}")
print(f"Superposition:   {model.cfg.num_features / model.cfg.hidden_dim:.1f}x")
assert model.hierarchy is not None
print(f"Hierarchical:    {len(model.hierarchy.feature_indices_used)}")
print(f"Free features:   {NUM_FEATURES - len(model.hierarchy.feature_indices_used)}")

Features:        512
Hidden dim:      128
Superposition:   4.0x
Hierarchical:    148
Free features:   364


## Exploring the labelled hierarchy

Each `HierarchyNode` now carries the `label`, `alpha`, and `beta` from the JSON.
Let's print the first tree to see the structure.

In [5]:
from sae_lens.synthetic.hierarchy.node import HierarchyNode


def print_tree(node: HierarchyNode, indent: int = 0) -> None:
    me_flag = " [ME]" if node.mutually_exclusive_children and node.children else ""
    label = node.label or f"feature_{node.feature_index}"
    print(
        f"{'  ' * indent}[{node.feature_index:3d}] {label}"
        f"  α={node.alpha:.2f} β={node.beta:.3f}{me_flag}"
    )
    for child in node.children:
        print_tree(child, indent + 1)


print("First tree in the hierarchy:")
print_tree(model.hierarchy.roots[0])

First tree in the hierarchy:
[  0] Adversarial Evasion  α=0.00 β=1.000 [ME]
  [  1] Physical Domain Evasion  α=0.65 β=0.760
    [  2] Adversarial Patch  α=0.65 β=0.760
    [  3] Physical World Perturbation  α=0.55 β=0.835
    [  4] Stop Sign Attack  α=0.50 β=0.866
  [  5] Digital Domain Evasion  α=0.55 β=0.835
    [  6] White-Box Evasion  α=0.65 β=0.760
    [  7] Black-Box Evasion  α=0.55 β=0.835
    [  8] Transfer-Based Evasion  α=0.50 β=0.866
  [  9] Semantic Evasion  α=0.45 β=0.893
    [ 10] Paraphrase Attack  α=0.65 β=0.760
    [ 11] Synonym Substitution  α=0.55 β=0.835


In [6]:
# Summary statistics across all trees
roots = model.hierarchy.roots


def tree_depth_node(node: HierarchyNode) -> int:
    if not node.children:
        return 0
    return 1 + max(tree_depth_node(c) for c in node.children)


max_d = max(tree_depth_node(r) for r in roots)
print(f"Root nodes:            {len(roots)}")
print(f"Hierarchical features: {len(model.hierarchy.feature_indices_used)}")
print(f"Max tree depth:        {max_d}")
print()

print("Root labels:")
for r in roots:
    size = len(r.get_all_feature_indices())
    print(f"  [{r.feature_index:3d}] {r.label}  ({size} nodes)")

Root nodes:            13
Hierarchical features: 148
Max tree depth:        2

Root labels:
  [  0] Adversarial Evasion  (12 nodes)
  [ 12] Adversarial Perturbation Crafting  (12 nodes)
  [ 24] Data Poisoning  (12 nodes)
  [ 36] Backdoor and Trojan Attacks  (11 nodes)
  [ 47] Model Extraction and Stealing  (11 nodes)
  [ 58] Membership Inference  (11 nodes)
  [ 69] Model Inversion and Data Reconstruction  (11 nodes)
  [ 80] Prompt Injection and LLM Exploitation  (13 nodes)
  [ 93] ML Reconnaissance  (11 nodes)
  [104] ML Supply Chain Attack  (12 nodes)
  [116] Adversarial Robustness Degradation  (11 nodes)
  [127] ML Model Access and Initial Compromise  (11 nodes)
  [138] ML Exfiltration  (10 nodes)


## Verifying the geometric properties

The key promise of semantic dictionaries is that $\cos(d_{\text{child}}, d_{\text{parent}}) = \alpha$
for every parent–child pair in the hierarchy. Let's verify this empirically across all
parent–child pairs.

In [7]:
import pytest  # noqa: F401 — only for approx; remove if not installed

W = model.feature_dict.feature_vectors.detach()  # (num_features, hidden_dim)
W_norm = W / W.norm(dim=1, keepdim=True).clamp(min=1e-8)

errors = []


def check_geometry(node: HierarchyNode) -> None:
    for child in node.children:
        cos_sim = (W_norm[node.feature_index] @ W_norm[child.feature_index]).item()
        expected = child.alpha
        errors.append(abs(cos_sim - expected))
        check_geometry(child)


for root in model.hierarchy.roots:
    check_geometry(root)

max_err = max(errors)
mean_err = sum(errors) / len(errors)

print(f"Parent–child pairs checked: {len(errors)}")
print(f"Max |cos_sim - alpha|:      {max_err:.2e}")
print(f"Mean |cos_sim - alpha|:     {mean_err:.2e}")

assert max_err < 1e-4, f"Geometric property violated: max error = {max_err}"
print("\nAll parent–child cosine similarities match their alpha values. ✓")

Parent–child pairs checked: 135
Max |cos_sim - alpha|:      1.43e-07
Mean |cos_sim - alpha|:     4.60e-08

All parent–child cosine similarities match their alpha values. ✓


## Visualising cosine similarities

### Parent–child vs random pairs

The semantic dictionary places related concepts closer together in feature space.
Let's compare the distribution of cosine similarities for:
- **Parent–child pairs** (connected in the hierarchy)
- **Random pairs** (unrelated features)

In [8]:
import plotly.graph_objects as go

# Collect all parent-child cosine similarities
parent_child_cosims = []


def collect_cosims(node: HierarchyNode) -> None:
    for child in node.children:
        cos_sim = (W_norm[node.feature_index] @ W_norm[child.feature_index]).item()
        parent_child_cosims.append(cos_sim)
        collect_cosims(child)


for root in model.hierarchy.roots:
    collect_cosims(root)

# Random pairs (sample ~same number)
n = len(parent_child_cosims)
idx_a = torch.randint(0, NUM_FEATURES, (n,))
idx_b = torch.randint(0, NUM_FEATURES, (n,))
random_cosims = (W_norm[idx_a] * W_norm[idx_b]).sum(dim=1).cpu().tolist()

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=parent_child_cosims, name="Parent–child",
    opacity=0.7, nbinsx=30, marker_color="steelblue",
))
fig.add_trace(go.Histogram(
    x=random_cosims, name="Random pairs",
    opacity=0.7, nbinsx=30, marker_color="coral",
))
fig.update_layout(
    barmode="overlay",
    title="Cosine similarity: parent–child vs random feature pairs",
    xaxis_title="Cosine similarity",
    yaxis_title="Count",
)
fig.show()

### Cosine similarity heatmap for one tree

Let's zoom into a single tactic tree ("Adversarial Evasion") and visualize the
full pairwise cosine similarity matrix with labels.

In [9]:
import plotly.express as px

# Collect all nodes in the first tree, in BFS order
from collections import deque


def bfs_nodes(root: HierarchyNode) -> list[HierarchyNode]:
    queue: deque[HierarchyNode] = deque([root])
    result = []
    while queue:
        node = queue.popleft()
        result.append(node)
        queue.extend(node.children)
    return result


tree_nodes = bfs_nodes(model.hierarchy.roots[0])
indices = [n.feature_index for n in tree_nodes]
short_labels = [
    (n.label or f"f{n.feature_index}")[:30] for n in tree_nodes
]

W_tree = W_norm[indices]  # (n_nodes, hidden_dim)
cosim_matrix = (W_tree @ W_tree.T).cpu().numpy()

fig = px.imshow(
    cosim_matrix,
    x=short_labels,
    y=short_labels,
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    title=f'Pairwise cosine similarities: "{model.hierarchy.roots[0].label}" tree',
    aspect="auto",
)
fig.update_layout(width=700, height=650)
fig.show()

## Sampling activations

Let's sample some hidden activations and inspect the feature firing statistics.

In [10]:
hidden_acts, feature_acts = model.sample_with_features(batch_size=10_000)

print(f"Hidden activations shape:  {hidden_acts.shape}")
print(f"Feature activations shape: {feature_acts.shape}")

l0 = (feature_acts > 0).float().sum(dim=1).mean()
print(f"Average L0:                {l0:.1f}")
print(f"Hidden act L2 norm (mean): {hidden_acts.norm(dim=1).mean():.2f}")

Hidden activations shape:  torch.Size([10000, 128])
Feature activations shape: torch.Size([10000, 512])
Average L0:                2.7
Hidden act L2 norm (mean): 1.93


In [11]:
import plotly.express as px

firing_freqs = (feature_acts > 0).float().mean(dim=0).cpu()

fig = px.histogram(
    x=firing_freqs.numpy(),
    nbins=40,
    log_y=True,
    title="Feature firing frequency distribution",
    labels={"x": "Firing frequency", "y": "Feature count"},
)
fig.show()

### Hierarchical firing: children only fire when parents fire

One core property of the hierarchy is that a child feature can only fire when its parent
fires. Let's verify this: for each parent–child pair, every sample where the child is
active should also have the parent active.

In [12]:
violations = 0
pairs_checked = 0
active = feature_acts > 0  # (batch, num_features)


def check_parent_child_firing(node: HierarchyNode) -> None:
    global violations, pairs_checked
    for child in node.children:
        parent_active = active[:, node.feature_index]
        child_active = active[:, child.feature_index]
        # child fires without parent
        v = (child_active & ~parent_active).sum().item()
        violations += v
        pairs_checked += 1
        check_parent_child_firing(child)


for root in model.hierarchy.roots:
    check_parent_child_firing(root)

print(f"Parent–child pairs checked: {pairs_checked}")
print(f"Violations (child fires without parent): {violations}")
assert violations == 0, "Hierarchical firing violated!"
print("\nHierarchical firing constraint holds for all samples. ✓")

Parent–child pairs checked: 135
Violations (child fires without parent): 0

Hierarchical firing constraint holds for all samples. ✓


## Training a BatchTopK SAE

Now let's train a sparse autoencoder on the custom semantic model using `SyntheticSAERunner`.

With a 512-feature, 128-dim model and 10M training samples, this takes **~2 minutes on CPU**.

In [13]:
from sae_lens import BatchTopKTrainingSAEConfig, LoggingConfig
from sae_lens.synthetic import SyntheticSAERunner, SyntheticSAERunnerConfig

runner_cfg = SyntheticSAERunnerConfig(
    synthetic_model=cfg,  # the SyntheticModelConfig we defined above
    sae=BatchTopKTrainingSAEConfig(
        d_in=HIDDEN_DIM,
        d_sae=NUM_FEATURES,  # match the true number of features
        k=15,
    ),
    training_samples=10_000_000,
    batch_size=1024,
    lr=3e-4,
    eval_frequency=500,
    eval_samples=100_000,
    logger=LoggingConfig(log_to_wandb=False),
    device=device,
)

runner = SyntheticSAERunner(runner_cfg)
result = runner.run()

Training SAE:   0%|          | 0/10000000 [00:00<?, ?it/s]

## Evaluating the trained SAE

In [14]:
ev = result.final_eval
assert ev is not None

print("BatchTopK SAE — semantic dictionary model")
print(f"  Explained variance (R²): {ev.explained_variance:.4f}")
print(f"  MCC:                     {ev.mcc:.4f}")
print(f"  Uniqueness:              {ev.uniqueness:.4f}")
print(f"  F1:                      {ev.classification.f1_score:.4f}")
print(f"  Precision:               {ev.classification.precision:.4f}")
print(f"  Recall:                  {ev.classification.recall:.4f}")
print(f"  SAE L0:                  {ev.sae_l0:.1f}")
print(f"  True L0:                 {ev.true_l0:.1f}")
print(f"  Dead latents:            {ev.dead_latents}")
print(f"  Shrinkage:               {ev.shrinkage:.4f}")

BatchTopK SAE — semantic dictionary model
  Explained variance (R²): 0.9709
  MCC:                     0.7321
  Uniqueness:              0.7734
  F1:                      0.4751
  Precision:               0.3721
  Recall:                  0.7830
  SAE L0:                  15.0
  True L0:                 2.7
  Dead latents:            72
  Shrinkage:               0.9847


### How well does the SAE recover the semantic structure?

For each ground-truth parent–child pair, we can ask: does the SAE's recovered
representation preserve the expected cosine similarity (alpha) between the matched
latents?

In [15]:
from sae_lens.synthetic import eval_sae_on_synthetic_data

# Re-run eval to get the matched feature mapping
eval_result = eval_sae_on_synthetic_data(
    sae=result.sae,
    feature_dict=model.feature_dict,
    activations_generator=model.activation_generator,
    num_samples=100_000,
    batch_size=1024,
)

print(f"MCC: {eval_result.mcc:.4f}")
print(f"F1:  {eval_result.classification.f1_score:.4f}")

MCC: 0.7321
F1:  0.4739


## Loading semantics in code

You can use the `load_semantic_dictionary` function to parse the JSON directly
into `ConceptNode` objects — useful for inspecting the taxonomy outside of a
full `SyntheticModel` build.

In [16]:
from sae_lens.synthetic import ConceptNode, load_semantic_dictionary

concept_roots: list[ConceptNode] = load_semantic_dictionary(str(JSON_PATH))

print(f"Loaded {len(concept_roots)} root concepts")
print()

# Inspect the first root
root = concept_roots[0]
print(f"Root: '{root.label}'  ME_children={root.mutually_exclusive_children}")
for child in root.children:
    print(f"  Child: '{child.label}'  α={child.alpha}  β={child.beta}")
    for grandchild in child.children:
        print(f"    Leaf: '{grandchild.label}'  α={grandchild.alpha}  β={grandchild.beta}")

Loaded 13 root concepts

Root: 'Adversarial Evasion'  ME_children=True
  Child: 'Physical Domain Evasion'  α=0.65  β=0.76
    Leaf: 'Adversarial Patch'  α=0.65  β=0.76
    Leaf: 'Physical World Perturbation'  α=0.55  β=0.835
    Leaf: 'Stop Sign Attack'  α=0.5  β=0.866
  Child: 'Digital Domain Evasion'  α=0.55  β=0.835
    Leaf: 'White-Box Evasion'  α=0.65  β=0.76
    Leaf: 'Black-Box Evasion'  α=0.55  β=0.835
    Leaf: 'Transfer-Based Evasion'  α=0.5  β=0.866
  Child: 'Semantic Evasion'  α=0.45  β=0.893
    Leaf: 'Paraphrase Attack'  α=0.65  β=0.76
    Leaf: 'Synonym Substitution'  α=0.55  β=0.835


## Bringing your own taxonomy

To use a custom taxonomy, create a JSON file with the following structure and pass
its path to `HierarchyConfig(semantic_dictionary_path=...)`:

```json
{
  "description": "My custom taxonomy",
  "trees": [
    {
      "label": "Animal",
      "alpha": 0.0,
      "beta": 1.0,
      "mutually_exclusive_children": true,
      "children": [
        {
          "label": "Dog",
          "alpha": 0.7,
          "beta": 0.714,
          "children": [
            { "label": "Poodle", "alpha": 0.8, "beta": 0.6, "children": [] },
            { "label": "Labrador", "alpha": 0.75, "beta": 0.661, "children": [] }
          ]
        },
        {
          "label": "Cat",
          "alpha": 0.6,
          "beta": 0.8,
          "children": [
            { "label": "Siamese", "alpha": 0.8, "beta": 0.6, "children": [] }
          ]
        }
      ]
    }
  ]
}
```

**Guidelines for α and β values:**
- Must satisfy $\alpha^2 + \beta^2 = 1$
- Typical ranges: α ∈ [0.3, 0.8] for semantically related concepts
- Root nodes use α=0.0, β=1.0 (no parent)
- Higher α = child is more similar to its parent

Features not defined in the JSON are automatically assigned orthogonalized random vectors.

## Summary

In this tutorial we covered:

1. **Loading a JSON taxonomy** with `load_semantic_dictionary()` and inspecting its structure
2. **Building a `SyntheticModel`** with semantic feature geometry by setting
   `HierarchyConfig(semantic_dictionary_path=...)`
3. **Verifying the geometry**: $\cos(d_{\text{child}}, d_{\text{parent}}) = \alpha$ holds
   exactly for all parent–child pairs
4. **Verifying hierarchical firing**: children only fire when their parents fire
5. **Visualising** parent–child vs random pair cosine similarities and per-tree heatmaps
6. **Training a BatchTopK SAE** on the custom model and evaluating with MCC/F1

### Next steps

- Edit `feature_hierarchies/mitre_atlas_adversarial_ml.json` to adjust α values and
  observe how feature recovery changes
- Create your own taxonomy JSON and build a domain-specific benchmark
- Compare SAE architectures (Matryoshka, Standard L1) on a fixed semantic dictionary
  to study which architecture best recovers hierarchical structure
- See the [semantic dictionary docs](https://decoderesearch.github.io/SAELens/semantic_dictionary/)
  for the full API reference and schema specification